# Evaluation 1:
HUMMUS Recipe Retrieval via Vector Store & LLM.

- For how to setup the vector store, refer to the `vector_store_food_RAG.ipynb` notebook.
- For how to setup the graph store, refer to the `init_graph_store_1.ipynb` and the `init_graph_store_2.ipynb` notebooks.
- On how to generate the question-answer pairs, refer to the `qa_generation.ipynb` notebook.


In [ ]:
import nest_asyncio
from llama_index.core import PropertyGraphIndex
from llama_index.core import Settings
from llama_index.core import StorageContext
from llama_index.core import VectorStoreIndex
from llama_index.core.indices.knowledge_graph import KGTableRetriever
from llama_index.embeddings.litellm import LiteLLMEmbedding
from llama_index.graph_stores.neo4j import Neo4jGraphStore
from llama_index.llms.litellm import LiteLLM
from pathlib import Path
from llama_index.core import load_index_from_storage

from src.food_rag.evaluation import retriever_evaluator
from src.food_rag.evaluation.retriever_evaluator import RUN_PATH
from src.food_rag.retriever import advanced_retriever
from src.food_rag.retriever import db_retriever
from src.food_rag.retriever import kg_retriever
from src.food_rag.retriever import pandas_retriever
from src.food_rag.retriever import wrapper_retriever
from src.food_rag.utils import qa_helper, embedding_helper, hummus_docify, postgres_helper
from src.food_rag.utils.embedding_helper import QA_SUB_PATH
from src.hummus_transformation import transform_kg
from src.persona import qa_generator
from src.utils import hummus_util
from src.utils import load_util

nest_asyncio.apply()  # Required to run the notebook in Jupyter for disabling async operations in the notebook

In [ ]:
## Output Paths
data_path = './data'
embedding_path = data_path + '/embeddings/snowflake_l'
qa_name = 'small_1'
run_name = 'eval_1'

# Load configs
nutrition_config = load_util.get_config('config/nutrition_config.yml')
persona_config = load_util.get_config('config/persona_config.yml')

# Load key
api_key = load_util.load_file('config/api_key')

In [ ]:
# Init LLM
llm = LiteLLM(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model="openai/llama3.1",
    max_retries=100,
    cooldown_time=60,
)

embed_model = LiteLLMEmbedding(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model_name="openai/Snowflake/snowflake-arctic-embed-l",
)

# Set Global Settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

In [ ]:
# Postgres credentials
postgres_credentials = {
    'db_name': 'rag_postgres',
    'host': 'localhost',
    'password': 'postgres',
    'port': '5432',
    'user': 'postgres',
}

# Connect to the default 'postgres' database
vector_store = postgres_helper.PostgresHelper(postgres_credentials, reset=False).connect(table_name='hummus_recipes',
                                                                                         embed_dim=1024)
vector_store_index = VectorStoreIndex.from_vector_store(vector_store)

In [ ]:
# Neo4j credentials
transform_mode = transform_kg.MODE_ALL
username = "neo4j"
password = "neo4jfoodrag"
url = "bolt://localhost:7687"

# Connect to the Neo4j KG store
graph_store = Neo4jGraphStore(
    username=username,
    password=password,
    url=url,
    database=transform_mode,
)
kg_transformer = transform_kg.Neo4JTransformer.load(data_path=data_path, mode=transform_mode)
schema = kg_transformer.get_schema()

# Connect to the Neo4j KG store with the direct embedded docs
# graph_store_2 = Neo4jGraphStore(
#     username=username,
#     password=password,
#     url=url,
#     database='docs',
# )

In [ ]:
# Load the current recipes data
hummus_data = load_util.get_current_hummus(data_path)
recipes_df = hummus_data['recipes_df']

In [ ]:
# Load question-answer pairs & their embeddings (only the recipe qa's)
question_answer_sets = qa_generator.load_qa_set(data_path, sub_dir_name=qa_name, recipes_df=recipes_df)
qa_recipe_doc_dict, _ = embedding_helper.load_existing_batches(
    embedding_path + QA_SUB_PATH + qa_name + '/qa_recipe_documents')

# Clean the qa set and add the embeddings
qa_set = qa_helper.get_best_questions(question_answer_sets['recipe_question_answer_set'], discard_empty_qas=True)
qa_set = qa_helper.extend_qa_embeddings(qa_set, qa_recipe_doc_dict)

In [ ]:
# Set the default metrics
metrics = ["hit_rate", "mrr", "ndcg", "precision", "recall", "f1", "who_score", "fsa_score", "nutri_score",
           "who_delta_score", "fsa_delta_score", "nutri_delta_score"]
# We eval the retrieved result against the whole baseline. However, for all retrievers we made sure that only #similarity_top_k recipes are retrieved.

# Set other default params
similarity_top_k = 10

In [ ]:
# Get dataset general nutrition scores
valid_nutrition_scores = qa_helper.get_qa_nutrition_scores(qa_set, recipes_df)
valid_nutrition_scores

## Retriever 1: Dense Vector Store Retriever
Just uses embeddings to retrieve the most similar recipes.

In [ ]:
# Set up the retriever
retriever_1 = db_retriever.RecipeRetriever(index=vector_store_index, recipes_df=recipes_df,
                                           vector_store_query_mode="default", similarity_top_k=similarity_top_k,
                                           split_node_ratio=2)

# Set up the evaluator
evaluator_1 = retriever_evaluator.DataFrameRetrieverEvaluator.from_metric_names(
    metric_names=metrics, retriever=retriever_1, recipes_df=recipes_df
)

# Evaluate
response_1, metrics_1 = evaluator_1.evaluate_dataset(qa_set=qa_set, data_path=data_path,
                                                     sub_dir_name=run_name + '/retriever_1', batch_size=10,
                                                     show_progress=True)
print(metrics_1)

## Retriever 2: Sparse Vector Store Retriever
Retrieves the most similar recipes using sparse embeddings by using BM25.


In [ ]:
# Set up the retriever
retriever_2 = db_retriever.RecipeRetriever(index=vector_store_index, recipes_df=recipes_df,
                                           vector_store_query_mode="sparse", similarity_top_k=similarity_top_k,
                                           split_node_ratio=2)

# Set up the evaluator
evaluator_2 = retriever_evaluator.DataFrameRetrieverEvaluator.from_metric_names(
    metric_names=metrics, retriever=retriever_2, recipes_df=recipes_df
)

# Evaluate
response_2, metrics_2 = evaluator_2.evaluate_dataset(qa_set=qa_set, data_path=data_path,
                                                     sub_dir_name=run_name + '/retriever_2', batch_size=10,
                                                     show_progress=True)
print(metrics_2)

## Retriever 3: Hybrid Vector Store Retriever


In [ ]:
# Set up the retriever
retriever_3 = db_retriever.RecipeRetriever(index=vector_store_index, recipes_df=recipes_df,
                                           vector_store_query_mode="hybrid", similarity_top_k=similarity_top_k,
                                           split_node_ratio=2)

# Set up the evaluator
evaluator_3 = retriever_evaluator.DataFrameRetrieverEvaluator.from_metric_names(
    metric_names=metrics, retriever=retriever_3, recipes_df=recipes_df
)

# Evaluate
response_3, metrics_3 = evaluator_3.evaluate_dataset(qa_set=qa_set, data_path=data_path,
                                                     sub_dir_name=run_name + '/retriever_3', batch_size=10,
                                                     show_progress=True)
print(metrics_3)

## Retriever 4: Text Search Vector Store Retriever


In [ ]:
# Set up the retriever
retriever_4 = db_retriever.RecipeRetriever(index=vector_store_index, recipes_df=recipes_df,
                                           vector_store_query_mode="text_search", similarity_top_k=similarity_top_k,
                                           split_node_ratio=2)

# Set up the evaluator
evaluator_4 = retriever_evaluator.DataFrameRetrieverEvaluator.from_metric_names(
    metric_names=metrics, retriever=retriever_4, recipes_df=recipes_df
)

# Evaluate
response_4, metrics_4 = evaluator_4.evaluate_dataset(qa_set=qa_set, data_path=data_path,
                                                     sub_dir_name=run_name + '/retriever_4', batch_size=10,
                                                     show_progress=True)
print(metrics_4)

## Retriever 5: Text-to-Pandas Retriever

In [ ]:
# Clean df for the pandas retriever
cleaned_recipes_df = recipes_df.copy()
cleaned_recipes_df = hummus_docify.stringify_nested_lists(cleaned_recipes_df)
cleaned_recipes_df = hummus_docify.stringify_nested_dicts(cleaned_recipes_df)
cleaned_recipes_df = cleaned_recipes_df.rename(
    columns={'duration': 'duration [min]', 'ingredients': 'ingredient_quantities',
             'ingredient_names': 'ingredients', 'direction_size': 'number_of_directions',
             'ingredients_sizes': 'number_of_ingredients'})
cleaned_recipes_df = cleaned_recipes_df.drop(
    columns=['last_changed_date', 'food_kg_locator', 'recipe_url', 'normalization_comment'])
cleaned_recipes_df = hummus_util.clean_serves_to_int(cleaned_recipes_df)

In [ ]:
# Set up the retriever
retriever_5 = pandas_retriever.PandasQueryRetriever(df=cleaned_recipes_df, similarity_top_k=similarity_top_k,
                                                    verbose=False)

# Set up the evaluator
evaluator_5 = retriever_evaluator.DataFrameRetrieverEvaluator.from_metric_names(
    metric_names=metrics, retriever=retriever_5, recipes_df=cleaned_recipes_df
)

# Evaluate
response_5, metrics_5 = evaluator_5.evaluate_dataset(qa_set=qa_set, data_path=data_path,
                                                     sub_dir_name=run_name + '/retriever_5', batch_size=10,
                                                     show_progress=True)
print(metrics_5)

## Retriever 6: Text-to-Cipher Retriever

In [ ]:
# Build the query engine
retriever_6 = kg_retriever.KGQueryEngine(
    recipes_df=recipes_df,
    similarity_top_k=similarity_top_k,
    storage_context=StorageContext.from_defaults(graph_store=graph_store),
    llm=llm,
    verbose=False,
    **schema
)

# Set up the evaluator
evaluator_6 = retriever_evaluator.DataFrameRetrieverEvaluator.from_metric_names(
    metric_names=metrics, retriever=retriever_6, recipes_df=recipes_df
)

# Evaluate
response_6, metrics_6 = evaluator_6.evaluate_dataset(qa_set=qa_set, data_path=data_path,
                                                     sub_dir_name=run_name + '/retriever_6', batch_size=10,
                                                     show_progress=True)
print(metrics_6)

## Retriever 7: Dense Neo4J Retriever

In [ ]:
# Set up the retriever
graph_store_2_index = load_index_from_storage(StorageContext.from_defaults(persist_dir=Path(data_path, 'neo4j_kg', 'docs')))

graph_store_2_retriever: KGTableRetriever = graph_store_2_index.as_retriever(similarity_top_k=similarity_top_k)
graph_store_2_retriever.num_chunks_per_query = 100
retriever_7 = wrapper_retriever.WrapperNodeRecipeRetriever(base_retriever=graph_store_2_retriever,
                                                           recipes_df=recipes_df)

# Set up the evaluator
evaluator_7 = retriever_evaluator.DataFrameRetrieverEvaluator.from_metric_names(
    metric_names=metrics, retriever=retriever_7, recipes_df=recipes_df
)

# Evaluate
response_7, metrics_7 = evaluator_7.evaluate_dataset(qa_set=qa_set, data_path=data_path,
                                                     sub_dir_name=run_name + '/retriever_7', batch_size=10,
                                                     show_progress=True)
print(metrics_7)

## Retriever 8: Fusion Retriever

In [ ]:
# Set up the retriever
fused_retrievers = [retriever_6, retriever_7]
retriever_8 = advanced_retriever.DataFrameFusionRetriever(retrievers=fused_retrievers, recipes_df=recipes_df,
                                                          mode="simple", num_queries=3, use_async=False,
                                                          similarity_top_k=similarity_top_k, verbose=True)

# Set up the evaluator
evaluator_8 = retriever_evaluator.DataFrameRetrieverEvaluator.from_metric_names(
    metric_names=metrics, retriever=retriever_8, recipes_df=recipes_df
)

# Evaluate
response_8, metrics_8 = evaluator_8.evaluate_dataset(qa_set=qa_set, data_path=data_path,
                                                     sub_dir_name=run_name + '/retriever_8', batch_size=10,
                                                     show_progress=True)
print(metrics_8)

## Retriever 9: Intersection Retriever

In [ ]:
# Set up the retriever
retriever_9 = advanced_retriever.IntersectionRetriever(retriever_5, retriever_7, verbose=True)

# Set up the evaluator
evaluator_9 = retriever_evaluator.DataFrameRetrieverEvaluator.from_metric_names(
    metric_names=metrics, retriever=retriever_9, recipes_df=recipes_df
)

# Evaluate
response_9, metrics_9 = evaluator_9.evaluate_dataset(qa_set=qa_set, data_path=data_path,
                                                     sub_dir_name=run_name + '/retriever_9', batch_size=10,
                                                     show_progress=True)
print(metrics_9)

## Retriever 10: Majority Intersection Retriever

In [ ]:
# Set up the retriever
maj_int_retrievers = [retriever_5, retriever_7, retriever_2]
vote_weights = {0: 0.3, 1: 0.3, 2: 0.3}  # Results must match at least 2 retrievers
retriever_10 = advanced_retriever.MajorityIntersectionRetriever(retrievers=maj_int_retrievers,
                                                                vote_weights=vote_weights, verbose=True)

# Set up the evaluator
evaluator_10 = retriever_evaluator.DataFrameRetrieverEvaluator.from_metric_names(
    metric_names=metrics, retriever=retriever_10, recipes_df=recipes_df
)

# Evaluate
response_10, metrics_10 = evaluator_10.evaluate_dataset(qa_set=qa_set, data_path=data_path,
                                                        sub_dir_name=run_name + '/retriever_10', batch_size=10,
                                                        show_progress=True)
print(metrics_10)

# Summarize and store the evaluation results

In [ ]:
# Summarize the evaluation results in a df
results_df = retriever_evaluator.summarize_results(
    {
        'Dense Vector Store Retriever (1)': metrics_1,
        'Sparse Vector Store Retriever (2)': metrics_2,
        'Hybrid Vector Store Retriever (3)': metrics_3,
        'Text Search Vector Store Retriever (4)': metrics_4,
        'Text-to-Pandas Retriever (5)': metrics_5,
        'Text-to-Cipher Retriever (6)': metrics_6,
        'Dense Neo4J Retriever (7)': metrics_7,
        'Fusion Retriever (8)': metrics_8,
        'Intersection Retriever (9)': metrics_9,
        'Majority Intersection Retriever (10)': metrics_10
    }
)
results_df

In [ ]:
# Store the evaluation results
load_util.store_data(data={
    'evaluation_results': results_df,
    'response_1': [response['retrieved_ids'] for response in response_1],
    'response_2': [response['retrieved_ids'] for response in response_2],
    'response_3': [response['retrieved_ids'] for response in response_3],
    'response_4': [response['retrieved_ids'] for response in response_4],
    'response_5': [response['retrieved_ids'] for response in response_5],
    'response_6': [response['retrieved_ids'] for response in response_6],
    'response_7': [response['retrieved_ids'] for response in response_7],
    'response_8': [response['retrieved_ids'] for response in response_8],
    'response_9': [response['retrieved_ids'] for response in response_9],
    'response_10': [response['retrieved_ids'] for response in response_10],
}, data_path=Path(data_path, RUN_PATH, run_name))